#### boot.s
- [el1_entry](https://youtu.be/UOpODhRuP3I?list=PLVxiWMqQvhg9FCteL7I0aohj1_YiUx1x8&t=629)
  - el1 address 會跳到這裡？
  - eret: 
    - 切換到指定的 Exception Level
    - 跳到指定的 PC
    - 而這兩件事是由兩個暫存器控制的：
    - SPSR_EL3	設定要切到哪個 EL + PSTATE
    - ELR_EL3	設定切換後要跳到哪個位址
    
```
    adr x0, el1_entry
    msr elr_el3, x0
    代表 eret 設定完成 exception level 後會跳回 el1_entry
```

- 清理 bss (全部設成 0 )

  - adr x0, bss_begin
  - adr x1, bss_end
  - sub x1, x1, x0
  - memzero(x0,x1): x0=bss_begin(start byte), x1 = bss_end-bss_begin = 要清多少 byte
#### mm.S

```
- memzero(x0,x1) , x0 start byte, x1 要清幾個 byte
- str xzr,[x0],#8  : post-increment addressing 
  - 等價於 : str xzr, [x0];  x0 addr += 8
  - *(uint64_t*)x0 = 0; x0+=8
  - 存 8 bytes 然後自動往後移動 8 
- subs x1, x1, #8
  - x1 = x1 - 8;
  - s 會 set flag NZCV
- b.gt memzero 
- 如果 x1 > 0 就跳回去
```

#### include/entry.h
#### src/entry.S
//Exception vectors table 
- ventry label, is just to jump (branch) to the label
- ventry sync_invalid-el1t : will jump to sync_invalid_el1t:
  - correspond to entry.h

```
sync_invalid_el1t:
	handle_invalid_entry  SYNC_INVALID_EL1t
```

- handle_invalid_entry macro  
  - will call `kernel_entry`
  - pass 3 param  x0,x1,x2   
  - branch  to show_invalid_entry_message(c code)
  - jump to err_hang

#### kernel exit  
- ventry	handle_el1_irq	
- handle_el1_irq:
	- `kernel_entry` 
	- bl	handle_irq  (c code)
	- kernel_exit 

#### kernel_entry  
- 注意 sub sp , store sp 是減少  (stack pointer, stack  往下長)
- when we get exception, we jump out our regular context 
  - so we need to store the state of the processor (so we can jump back and restore prevois state)
  - stp : store all of our register to stack pointer (sp)
#### kernel exit  
- ldp back from stack  , add sp 是增加  (stack pointer)


### irq.h 
### include/irq.S
### include/peripherals/irq.h
BCM2835 docs  page 113
- irq table  
- 29 is aux interrupt

```c
enum vc_irqs {
    AUX_IRQ = (1 << 29)
};
```

### src/irq.c
所以基本上我們做的是 
在 irq.c 掛上一個 handle_irq  
假如 aux_interrupt 的時候就會接收 uart recv printf

- BCM2835 docs  page 15
- AUM_MU_IIR_REG register
- 10: receiver holds valid byte